# **Task 06 — Machine Learning Model Evaluation & Assessment**
**Course:** CCS3440 - Artificial Intelligence  
**Project:** SmartCare AI Risk Prediction System  

---  
### **Objectives:**
- Comprehensive evaluation of classification models (Logistic Regression, Decision Tree, Random Forest)
- Generate Probability Predictions (`predict_proba`)
- Compute full evaluation metrics: **Accuracy, Precision, Recall, F1 Score, ROC-AUC**
- Generate detailed **Classification Reports** for all models
- Visualize **Confusion Matrices** for prediction errors analysis
- Plot comparative **ROC Curves** with Area Under Curve (AUC) scores
- Graphically compare all performance metrics
- Select and save the **Best Performing Model** (`best_model.pkl`)
- Export Task 06 summary files (`task06_evaluation_results.csv`, `task06_evaluation_percentage.csv`, `task06_hyperparameter_summary.csv`)

## **1. Import Required Libraries**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)

print("Libraries imported successfully for Task 06!")

## **2. Load Dataset & Preprocessing**

In [ ]:
try:
    from google.colab import files
    print("Upload smartcare_final_preprocessed_dataset.csv:")
    uploaded = files.upload()
except Exception:
    pass

# Load dataset
df = pd.read_csv("smartcare_final_preprocessed_dataset.csv")

# Separate Target and Features
y = df["no_show"]
X = df.drop(columns=["no_show"], errors="ignore")

# Handle missing values and categorical encoding
numerical_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object"]).columns

X = X.copy()
for column in numerical_features:
    X[column] = X[column].fillna(X[column].median())
for column in categorical_features:
    X[column] = X[column].fillna(X[column].mode()[0])

X_encoded = pd.get_dummies(X, columns=categorical_features, drop_first=True)
bool_columns = X_encoded.select_dtypes(include=["bool"]).columns
X_encoded[bool_columns] = X_encoded[bool_columns].astype(int)

# Train / Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.20,
    random_state=42
)

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data Preprocessing Complete!")
print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

## **3. Train Models for Evaluation**

In [ ]:
# Model 1: Logistic Regression
logistic_model = LogisticRegression(max_iter=1000, random_state=42)
logistic_model.fit(X_train_scaled, y_train)
logistic_predictions = logistic_model.predict(X_test_scaled)

# Model 2: Decision Tree
decision_tree_model = DecisionTreeClassifier(
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)
decision_tree_model.fit(X_train, y_train)
decision_tree_predictions = decision_tree_model.predict(X_test)

# Model 3: Random Forest
random_forest_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)
random_forest_model.fit(X_train, y_train)
random_forest_predictions = random_forest_model.predict(X_test)

print("All 3 models trained and predictions generated!")

## **4. Generate Probability Predictions**

In [ ]:
logistic_probabilities = logistic_model.predict_proba(X_test_scaled)[:, 1]
decision_tree_probabilities = decision_tree_model.predict_proba(X_test)[:, 1]
random_forest_probabilities = random_forest_model.predict_proba(X_test)[:, 1]

print("Probability predictions generated successfully!")

## **5. Calculate Evaluation Metrics**

In [ ]:
# Logistic Regression Metrics
logistic_accuracy = accuracy_score(y_test, logistic_predictions)
logistic_precision = precision_score(y_test, logistic_predictions, zero_division=0)
logistic_recall = recall_score(y_test, logistic_predictions, zero_division=0)
logistic_f1 = f1_score(y_test, logistic_predictions, zero_division=0)
logistic_roc_auc = roc_auc_score(y_test, logistic_probabilities)

# Decision Tree Metrics
decision_tree_accuracy = accuracy_score(y_test, decision_tree_predictions)
decision_tree_precision = precision_score(y_test, decision_tree_predictions, zero_division=0)
decision_tree_recall = recall_score(y_test, decision_tree_predictions, zero_division=0)
decision_tree_f1 = f1_score(y_test, decision_tree_predictions, zero_division=0)
decision_tree_roc_auc = roc_auc_score(y_test, decision_tree_probabilities)

# Random Forest Metrics
random_forest_accuracy = accuracy_score(y_test, random_forest_predictions)
random_forest_precision = precision_score(y_test, random_forest_predictions, zero_division=0)
random_forest_recall = recall_score(y_test, random_forest_predictions, zero_division=0)
random_forest_f1 = f1_score(y_test, random_forest_predictions, zero_division=0)
random_forest_roc_auc = roc_auc_score(y_test, random_forest_probabilities)

print("Evaluation metrics calculated for all models!")

## **6. Evaluation Results Table**

In [ ]:
evaluation_results = pd.DataFrame({
    "Model": ["Logistic Regression", "Decision Tree", "Random Forest"],
    "Accuracy": [logistic_accuracy, decision_tree_accuracy, random_forest_accuracy],
    "Precision": [logistic_precision, decision_tree_precision, random_forest_precision],
    "Recall": [logistic_recall, decision_tree_recall, random_forest_recall],
    "F1 Score": [logistic_f1, decision_tree_f1, random_forest_f1],
    "ROC-AUC": [logistic_roc_auc, decision_tree_roc_auc, random_forest_roc_auc]
})

evaluation_percentage = evaluation_results.copy()
metrics = ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"]
for metric in metrics:
    evaluation_percentage[metric] = (evaluation_percentage[metric] * 100).round(2)

print("--- Model Evaluation Results (Decimal) ---")
print(evaluation_results)
print("\n--- Model Evaluation Results (Percentage) ---")
print(evaluation_percentage)

## **7. Classification Reports**

In [ ]:
print("==========================================")
print("LOGISTIC REGRESSION - CLASSIFICATION REPORT")
print("==========================================")
print(classification_report(y_test, logistic_predictions, target_names=["No Show (0)", "Show (1)"], zero_division=0))

print("==========================================")
print("DECISION TREE - CLASSIFICATION REPORT")
print("==========================================")
print(classification_report(y_test, decision_tree_predictions, target_names=["No Show (0)", "Show (1)"], zero_division=0))

print("==========================================")
print("RANDOM FOREST - CLASSIFICATION REPORT")
print("==========================================")
print(classification_report(y_test, random_forest_predictions, target_names=["No Show (0)", "Show (1)"], zero_division=0))

## **8. Confusion Matrices for All Models**

In [ ]:
model_predictions = {
    "Logistic Regression": logistic_predictions,
    "Decision Tree": decision_tree_predictions,
    "Random Forest": random_forest_predictions
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (model_name, predictions) in zip(axes, model_predictions.items()):
    cm = confusion_matrix(y_test, predictions)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["No Show (0)", "Show (1)"], yticklabels=["No Show (0)", "Show (1)"], ax=ax)
    ax.set_title(f"Confusion Matrix - {model_name}")
    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("Actual Label")

plt.tight_layout()
plt.show()

## **9. ROC Curve Comparison**

In [ ]:
plt.figure(figsize=(9, 7))

# Logistic Regression
fpr_lr, tpr_lr, _ = roc_curve(y_test, logistic_probabilities)
plt.plot(fpr_lr, tpr_lr, label=f"Logistic Regression (AUC = {logistic_roc_auc:.3f})")

# Decision Tree
fpr_dt, tpr_dt, _ = roc_curve(y_test, decision_tree_probabilities)
plt.plot(fpr_dt, tpr_dt, label=f"Decision Tree (AUC = {decision_tree_roc_auc:.3f})")

# Random Forest
fpr_rf, tpr_rf, _ = roc_curve(y_test, random_forest_probabilities)
plt.plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC = {random_forest_roc_auc:.3f})")

# Reference line
plt.plot([0, 1], [0, 1], linestyle="--", label="Random Classifier")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison - Task 06")
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()

## **10. Graphical Metrics Comparison**

In [ ]:
plot_data = evaluation_results.set_index("Model")[["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"]]

plot_data.plot(kind="bar", figsize=(11, 6))
plt.title("Machine Learning Model Performance Comparison (Task 06)")
plt.xlabel("Model")
plt.ylabel("Score")
plt.ylim(0, 1.05)
plt.xticks(rotation=0)
plt.legend(title="Evaluation Metrics")
plt.tight_layout()
plt.show()

## **11. Find & Save Best Performing Model**

In [ ]:
best_model_row = evaluation_results.sort_values(by=["F1 Score", "ROC-AUC", "Accuracy"], ascending=False).iloc[0]
best_model_name = best_model_row["Model"]

if best_model_name == "Logistic Regression":
    best_model = logistic_model
elif best_model_name == "Decision Tree":
    best_model = decision_tree_model
elif best_model_name == "Random Forest":
    best_model = random_forest_model
else:
    raise ValueError("Unknown model selected.")

joblib.dump(best_model, "best_model.pkl")

print("==========================================")
print("BEST-PERFORMING MODEL SAVED SUCCESSFULLY")
print("==========================================")
print("Model:", best_model_name)
print(f"Accuracy : {best_model_row['Accuracy']:.4f}")
print(f"Precision: {best_model_row['Precision']:.4f}")
print(f"Recall   : {best_model_row['Recall']:.4f}")
print(f"F1 Score : {best_model_row['F1 Score']:.4f}")
print(f"ROC-AUC  : {best_model_row['ROC-AUC']:.4f}")
print("Saved as: best_model.pkl")

## **12. Hyperparameter Summary Table**

In [ ]:
hyperparameter_summary = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest"
    ],
    "Selected Hyperparameters": [
        "max_iter=1000, random_state=42",
        "max_depth=10, min_samples_split=5, min_samples_leaf=2, random_state=42",
        "n_estimators=200, max_depth=15, min_samples_split=5, min_samples_leaf=2, random_state=42"
    ]
})

print(hyperparameter_summary)

## **13. Save Task 06 Evaluation Results**

In [ ]:
evaluation_results.to_csv("task06_evaluation_results.csv", index=False)
evaluation_percentage.to_csv("task06_evaluation_percentage.csv", index=False)
hyperparameter_summary.to_csv("task06_hyperparameter_summary.csv", index=False)

print("Task 06 evaluation results saved successfully!")

## **14. Download Task 06 Artifacts (Google Colab)**

In [ ]:
try:
    from google.colab import files
    files.download("task06_evaluation_results.csv")
    files.download("task06_evaluation_percentage.csv")
    files.download("task06_hyperparameter_summary.csv")
    files.download("best_model.pkl")
except ImportError:
    print("Local environment detected - Files saved in current directory:")
    print("- best_model.pkl")
    print("- task06_evaluation_results.csv")
    print("- task06_evaluation_percentage.csv")
    print("- task06_hyperparameter_summary.csv")